# 🎯 PaliGemma Math Recognition Training (H100 Optimized)

**Streamlined training pipeline for Google Colab Pro with H100**

This notebook will:
1. Mount Google Drive for persistent storage
2. Install dependencies
3. Download/mount the MathWriting dataset
4. Train PaliGemma-3B with LoRA (~12-18 hours on H100)
5. Auto-save checkpoints to Drive

**Setup Requirements:**
- Google Colab Pro (for H100 access)
- HuggingFace token with PaliGemma access
- ~10GB free space on Google Drive

In [ ]:
# ============================================================================
# 1️⃣ SETUP: Mount Drive & Configure Environment
# ============================================================================

import os
from google.colab import drive, userdata

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Create working directory
WORK_DIR = '/content/drive/MyDrive/math-training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"✅ Working directory: {WORK_DIR}")
print("✅ All outputs will be saved to Google Drive (survives tab closures)")

In [ ]:
# Check GPU allocation
!nvidia-smi

import torch
print(f"\n🖥️  Device: {torch.cuda.get_device_name(0)}")
print(f"📊 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

if 'H100' not in torch.cuda.get_device_name(0):
    print("\n⚠️  WARNING: Not using H100. Go to Runtime > Change runtime type > H100 GPU")

In [ ]:
# Install dependencies
!pip install -q torch>=2.2.0 transformers>=4.50.0 peft>=0.15.0 \
    datasets huggingface_hub bitsandbytes>=0.45.0 \
    matplotlib pillow numpy tqdm timm

print("✅ Dependencies installed")

In [ ]:
# ============================================================================
# 2️⃣ AUTHENTICATION: HuggingFace Token
# ============================================================================

from huggingface_hub import login

# Get token from Colab secrets
# Go to 🔑 (left sidebar) → Add secret → Name: HF_TOKEN → Value: your_token
# Create token at: https://huggingface.co/settings/tokens
# Request access to: https://huggingface.co/google/paligemma-3b-pt-224

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace authenticated")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("\nPlease:")
    print("1. Create token: https://huggingface.co/settings/tokens")
    print("2. Request access: https://huggingface.co/google/paligemma-3b-pt-224")
    print("3. Add to Colab secrets (🔑 icon): Name=HF_TOKEN")
    raise

In [ ]:
# ============================================================================
# 3️⃣ DATASET: Download MathWriting-2024
# ============================================================================

import subprocess
import glob

DATA_DIR = f"{WORK_DIR}/mathwriting-2024"
DATASET_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"

# Check if already downloaded
if os.path.exists(f"{DATA_DIR}/train"):
    print(f"✅ Dataset already exists at {DATA_DIR}")
    
    # Show stats
    for split in ['train', 'valid', 'test']:
        count = len(glob.glob(f"{DATA_DIR}/{split}/*.inkml"))
        print(f"   {split}: {count:,} files")
else:
    print(f"📥 Downloading dataset (~2.9 GB, 5-10 minutes)...")
    
    # Download to Drive (persistent)
    !wget -q --show-progress {DATASET_URL} -O {WORK_DIR}/mathwriting-2024.tgz
    
    print("📦 Extracting...")
    !tar -xzf {WORK_DIR}/mathwriting-2024.tgz -C {WORK_DIR}
    
    # Cleanup
    !rm {WORK_DIR}/mathwriting-2024.tgz
    
    print("✅ Dataset ready!")
    for split in ['train', 'valid', 'test']:
        count = len(glob.glob(f"{DATA_DIR}/{split}/*.inkml"))
        print(f"   {split}: {count:,} files")

In [ ]:
# ============================================================================
# 4️⃣ PROJECT FILES: Clone from GitHub
# ============================================================================

# Clone repo if needed
if not os.path.exists('data_preprocessing.py'):
    print("📥 Cloning project files...")
    !git clone https://github.com/hudsonmp/realtime-math.git temp_repo
    !cp temp_repo/*.py .
    !rm -rf temp_repo
    print("✅ Project files ready")
else:
    print("✅ Project files already present")

# Verify
required = ['data_preprocessing.py', 'train.py']
for f in required:
    assert os.path.exists(f), f"Missing {f}"
    print(f"   ✅ {f}")

In [ ]:
# ============================================================================
# 5️⃣ TRAINING: PaliGemma-3B + LoRA (H100 Optimized)
# ============================================================================

import torch
from torch.utils.data import DataLoader
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, get_scheduler
from peft import LoraConfig, get_peft_model
from data_preprocessing import MathWritingDataset, LaTeXTokenizer
from tqdm import tqdm
import time

# ============================================================================
# HYPERPARAMETERS (Optimized for H100 - 12-18 hour training)
# ============================================================================

EPOCHS = 10
BATCH_SIZE = 16          # H100 can handle larger batches
GRAD_ACCUM = 2           # Effective batch size = 32
LEARNING_RATE = 2e-4     # Slightly higher for faster convergence
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Paths
CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = "cuda"

print("="*70)
print("🚀 TRAINING CONFIGURATION")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"LoRA Rank: {LORA_R}")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Dataset: {DATA_DIR}")
print("="*70)

# ============================================================================
# LOAD MODEL
# ============================================================================

print("\n📦 Loading PaliGemma-3B...")
processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224")

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map=None  # Load to CPU first for LoRA setup
)

# Add LoRA adapters
print("🔧 Applying LoRA...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Added more for better perf
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.to(device)

model.print_trainable_parameters()

# ============================================================================
# LOAD DATASETS
# ============================================================================

print("\n📊 Loading datasets...")
train_ds = MathWritingDataset(DATA_DIR, split='train')
valid_ds = MathWritingDataset(DATA_DIR, split='valid')

print(f"   Train: {len(train_ds):,} samples")
print(f"   Valid: {len(valid_ds):,} samples")

latex_tokenizer = LaTeXTokenizer()

def collate_fn(batch):
    """Collate function for DataLoader."""
    stroke_texts = [item['stroke_text'] for item in batch]
    images = [item['image'] for item in batch]
    labels = [item['label'] for item in batch]

    # Process inputs
    inputs = processor(
        text=stroke_texts,
        images=images,
        padding="longest",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )

    # Process labels
    label_encodings = processor.tokenizer(
        labels,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # Create labels tensor
    batch_size = inputs['input_ids'].shape[0]
    seq_length = inputs['input_ids'].shape[1]
    labels_tensor = torch.full((batch_size, seq_length), -100, dtype=torch.long)

    for i, label_ids in enumerate(label_encodings['input_ids']):
        label_length = (label_ids != processor.tokenizer.pad_token_id).sum().item()
        labels_tensor[i, -label_length:] = label_ids[:label_length]

    inputs['labels'] = labels_tensor
    return inputs

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,  # Lower for Colab stability
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# ============================================================================
# TRAINING SETUP
# ============================================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

num_training_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
scheduler = get_scheduler(
    "cosine",  # Cosine annealing for better convergence
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps
)

print(f"\n📈 Training Setup:")
print(f"   Steps per epoch: {len(train_loader)}")
print(f"   Total optimization steps: {num_training_steps}")
print(f"   Warmup steps: {WARMUP_STEPS}")

# ============================================================================
# TRAINING LOOP
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

model.train()
global_step = 0
best_cer = float('inf')
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"📅 EPOCH {epoch + 1}/{EPOCHS}")
    print('='*70)
    
    epoch_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        
        # Gradient accumulation
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        epoch_loss += loss.item() * GRAD_ACCUM
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item() * GRAD_ACCUM:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })
    
    avg_loss = epoch_loss / len(train_loader)
    elapsed = (time.time() - start_time) / 3600
    
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {avg_loss:.4f}")
    print(f"   Time Elapsed: {elapsed:.2f} hours")
    
    # ========================================================================
    # VALIDATION
    # ========================================================================
    
    print("\n🔍 Running validation...")
    model.eval()
    val_loss = 0
    total_cer = 0
    num_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Validation"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            labels = batch['labels'].to(device)
            
            outputs = model(**{**inputs, 'labels': labels})
            val_loss += outputs.loss.item()
            
            # Generate predictions
            generated = model.generate(**inputs, max_length=64)
            
            # Compute CER
            for pred_ids, label_ids in zip(generated, labels):
                pred_text = processor.decode(pred_ids, skip_special_tokens=True)
                label_text = processor.decode(label_ids[label_ids != -100], skip_special_tokens=True)
                cer = latex_tokenizer.compute_cer(pred_text, label_text)
                total_cer += cer
                num_samples += 1
    
    avg_val_loss = val_loss / len(valid_loader)
    avg_cer = total_cer / num_samples if num_samples > 0 else 0
    
    print(f"\n📊 Validation Results:")
    print(f"   Val Loss: {avg_val_loss:.4f}")
    print(f"   CER: {avg_cer:.4f}")
    
    model.train()
    
    # ========================================================================
    # CHECKPOINTING
    # ========================================================================
    
    # Save best model
    if avg_cer < best_cer:
        print(f"\n🎉 New best CER: {best_cer:.4f} → {avg_cer:.4f}")
        best_cer = avg_cer
        save_path = f"{CHECKPOINT_DIR}/best_model"
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        print(f"✅ Best model saved to {save_path}")
    
    # Save periodic checkpoint
    if (epoch + 1) % 2 == 0:
        save_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
        model.save_pretrained(save_path)
        print(f"💾 Checkpoint saved: {save_path}")

# ============================================================================
# FINAL SAVE
# ============================================================================

final_path = f"{CHECKPOINT_DIR}/final_model"
model.save_pretrained(final_path)
processor.save_pretrained(final_path)

total_time = (time.time() - start_time) / 3600

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"Total time: {total_time:.2f} hours")
print(f"Best CER: {best_cer:.4f}")
print(f"\nModels saved to: {CHECKPOINT_DIR}")
print("   - best_model/     (lowest CER)")
print("   - final_model/    (last epoch)")
print("   - epoch_X/        (periodic checkpoints)")
print("\n✅ All files saved to Google Drive (persistent)")
print("="*70)

In [ ]:
# ============================================================================
# 6️⃣ OPTIONAL: Test Inference
# ============================================================================

# Load best model for testing
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
import torch

print("Loading best model for inference...")

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    f"{CHECKPOINT_DIR}/best_model"
)

processor = AutoProcessor.from_pretrained(f"{CHECKPOINT_DIR}/best_model")

print("✅ Model loaded!")

# Test on a sample
from data_preprocessing import MathWritingDataset

test_ds = MathWritingDataset(DATA_DIR, split='test')
sample = test_ds[0]

inputs = processor(
    text=sample['stroke_text'],
    images=sample['image'],
    return_tensors="pt"
).to("cuda")

generated = model.generate(**inputs, max_length=64)
prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\n🧪 Sample Test:")
print(f"   Ground Truth: {sample['label']}")
print(f"   Prediction:   {prediction}")

---

## 📝 Notes

**Training Time (H100):**
- Expected: 12-18 hours for 10 epochs
- ~1.2-1.8 hours per epoch

**Colab Pro Features:**
- ✅ Survives tab closures
- ✅ Checkpoints saved to Google Drive
- ✅ Can reconnect and resume

**Checkpoints:**
- Best model saved based on validation CER
- Periodic saves every 2 epochs
- All LoRA adapters (~27M params, not full 3B model)

**To Resume Training:**
If disconnected, load the latest checkpoint:
```python
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "path/to/checkpoint")
```